In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time,math

In [6]:
def scaled_dot_product_attention(q,k,v,mask=None):
  #q:[B,H,T,D_Q]
  #k:[B,H,t,D_K]
  #v:[B,H,T,D_V]
  d_k=q.size(-1)
  scores=q@k.transpose(-2,-1)/math.sqrt(d_k)
  if mask is not None:
    scores=scores.masked_fill(~mask,float('-inf'))
  att=F.softmax(scores,dim=-1)
  return scores@v,att

q=torch.randn(2,5,7,8)
k=torch.randn(2,5,7,8)
v=torch.randn(2,5,7,8)
scores,_=scaled_dot_product_attention(q,k,v)
print(scores)


tensor([[[[-2.4853e-01, -1.8215e+00,  3.1196e-01,  3.3267e-02, -2.2466e-01,
           -6.7793e-01, -1.4235e-01, -6.9656e-01],
          [-4.4472e+00,  3.7528e+00, -1.1923e+00,  1.6329e-01, -5.6532e+00,
           -2.0530e+00, -3.6501e+00, -8.7600e-01],
          [-2.4779e+00,  6.3910e+00, -2.3495e+00,  3.8192e-01, -2.5293e+00,
           -2.5913e-01, -3.2388e+00, -1.9381e-01],
          [-4.2004e+00,  4.3188e+00, -8.9547e-01,  1.0219e+00, -2.4603e+00,
           -3.5589e-01, -2.1235e+00, -1.1152e+00],
          [ 1.3369e+00,  1.2414e+00, -2.7306e-01, -1.1507e+00, -1.1404e+00,
           -5.0144e-01, -3.5813e-01,  1.4314e+00],
          [-8.2224e-01,  1.7740e+00, -4.7257e+00,  3.7375e-01, -3.5447e+00,
           -1.0157e+00, -5.0090e+00, -2.5068e+00],
          [ 2.5559e+00, -3.4973e+00,  8.2021e-01, -4.1733e-02,  3.5969e+00,
            4.2713e-01,  1.7429e+00, -1.8085e-01]],

         [[-2.4463e+00, -2.0518e-01,  2.9621e+00,  2.7030e+00,  2.9559e+00,
           -2.5735e+00,  5.9989e-

In [14]:
def my_softmax(x,dim=-1):
  max_val=torch.max(x,dim=dim,keepdim=True).values
  print("max_val:",max_val)
  exp_x=torch.exp(x-max_val)
  print(x-max_val)
  print("exp_x:",exp_x)
  sum_exp=torch.sum(exp_x,dim=dim,keepdim=True)
  print("sum_exp:",sum_exp)
  return exp_x/sum_exp
x=torch.tensor([[1,2],[3,4]])
out=my_softmax(x)
print(out)



max_val: tensor([[2],
        [4]])
tensor([[-1,  0],
        [-1,  0]])
exp_x: tensor([[0.3679, 1.0000],
        [0.3679, 1.0000]])
sum_exp: tensor([[1.3679],
        [1.3679]])
tensor([[0.2689, 0.7311],
        [0.2689, 0.7311]])


In [16]:
def my_softmax_2(x,dim=-1):
  exp_x=torch.exp(x)
  sum_exp=torch.sum(exp_x,dim=dim,keepdim=True)
  return exp_x/sum_exp
x=torch.tensor([[1,2],[3,4]])
print(my_softmax_2(x))

tensor([[0.2689, 0.7311],
        [0.2689, 0.7311]])


In [17]:
import torch

def softmax_naive(x, dim=-1):
    e = torch.exp(x)
    return e / e.sum(dim=dim, keepdim=True)

def softmax_stable(x, dim=-1):
    e = torch.exp(x - x.max(dim=dim, keepdim=True).values)
    return e / e.sum(dim=dim, keepdim=True)

# 1) 上溢：数值很大
x = torch.tensor([[1000., 1001.]])
print(torch.exp(x))          # tensor([[inf, inf]])
print(softmax_naive(x))      # tensor([[nan, nan]])  因为 inf/inf
print(softmax_stable(x))     # tensor([[0.2689, 0.7311]])  正确

# 2) 下溢：数值很负
x = torch.tensor([[-1000., -1001.]])
print(torch.exp(x))          # tensor([[0., 0.]])
print(softmax_naive(x))      # tensor([[nan, nan]])  因为 0/0
print(softmax_stable(x))     # tensor([[0.7311, 0.2689]])  正确

# 3) 临界点：float32 大约在 exp(88.7) 就溢出
print(torch.exp(torch.tensor(88.)))   # 1.6516e+38，还能表示
print(torch.exp(torch.tensor(89.)))   # inf

tensor([[inf, inf]])
tensor([[nan, nan]])
tensor([[0.2689, 0.7311]])
tensor([[0., 0.]])
tensor([[nan, nan]])
tensor([[0.7311, 0.2689]])
tensor(1.6516e+38)
tensor(inf)


In [23]:
#完整的放缩点击包含softmax的完整代码：
def my_softmax_3(x,dim=-1):
  max_val=torch.max(x,dim=dim,keepdim=True).values
  exp_x=torch.exp(x-max_val)
  return exp_x/torch.sum(exp_x,dim=dim,keepdim=True)
def scaled_dot_product_attention_1(q,k,v,mask=None):
  #q=[B,H,t,d_k],k=[B,H,t,d_k],v=[B,H,t,d_v]
  d_k=q.size(-1)
  scores=q@k.transpose(-2,-1)/math.sqrt(d_k)
  if mask is not None:
    scores=scores.masked_fill(~mask,float("-inf"))
  att=my_softmax_3(scores)
  return att@v,att
q=torch.randn(2,5,7,8)
k=torch.randn(2,5,7,8)
v=torch.randn(2,5,7,9)
T=7
mask=torch.tril(torch.ones(T,T,dtype=bool))
print("mask:",mask)
out,att=scaled_dot_product_attention_1(q,k,v)
print("out:",out)
print("att:",att)
mask_out,mask_att=scaled_dot_product_attention_1(q,k,v,mask)
print("causal_out:",mask_out)
print("causal_att",mask_att)

mask: tensor([[ True, False, False, False, False, False, False],
        [ True,  True, False, False, False, False, False],
        [ True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False],
        [ True,  True,  True,  True,  True, False, False],
        [ True,  True,  True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True,  True,  True]])
out: tensor([[[[ 1.5395e-01, -6.3432e-01, -5.8920e-01, -2.6498e-01, -2.8884e-01,
           -1.3824e-01,  9.4913e-01, -2.8184e-01,  1.2655e-01],
          [ 5.8910e-01, -2.1034e-01, -5.9432e-01,  3.7608e-01, -6.1183e-01,
            3.6468e-02,  6.9237e-01, -7.1844e-01, -3.6435e-01],
          [ 1.9328e-01, -5.2376e-01, -3.4848e-01,  1.6627e-01, -4.1448e-01,
            2.0262e-01,  7.5780e-01, -4.5192e-01, -5.9607e-01],
          [ 5.8302e-01, -7.0487e-01, -7.0642e-01,  2.6979e-02, -2.7794e-01,
           -2.0079e-01,  3.1290e-01, -3.2101e-01, -3.5339e-01],
          [ 2.6

In [32]:
class MutleHeadAttention(nn.Module):
  def __init__(self,d_model,n_head,drop_out=0.01) -> None:
    super().__init__()
    assert d_model%n_head==0
    self.d_model=d_model
    self.h=n_head
    self.dk=d_model//n_head
    self.W_q=nn.Linear(d_model,d_model)
    self.W_k=nn.Linear(d_model,d_model)
    self.W_v=nn.Linear(d_model,d_model)
    self.W_o=nn.Linear(d_model,d_model)
    self.dropout=nn.Dropout(drop_out)


  def split(self,x):
    B,T,_=x.shape
    return x.view(B,T,self.h,self.dk).transpose(1,2)

  def forward(self,x_q,x_kv,mask=None):
    q=self.split(self.W_q(x_q))
    k=self.split(self.W_k(x_kv))
    v=self.split(self.W_v(x_kv))
    out,self.lastatt=scaled_dot_product_attention_1(q,k,v,mask)
    out=self.dropout(out)
    B,H,T,dk=out.shape
    out=out.transpose(1,2).contiguous().view(B,T,H*dk)
    return self.W_o(out)


mha=MutleHeadAttention(32,4)
mha.eval() # Disable dropout for deterministic evaluation
x=torch.randn(2,6,32)
causal=torch.tril(torch.ones(6,6,dtype=bool))[None, None]
y1=mha(x,x,causal)
x2=x.clone();x2[:,4:]=torch.randn(2,2,32)
y2=mha(x2,x2,causal)
print(y1.shape)
assert torch.allclose(y1[:, :4], y2[:, :4], atol=1e-6)
print("causality OK")

torch.Size([2, 6, 32])
causality OK


In [38]:
class multiheadattention(nn.Module):
  def __init__(self,d_model,n_head,drop_out=0.01) -> None:
    super().__init__()
    assert d_model%n_head==0
    self.d_model=d_model
    self.h=n_head
    self.dk=d_model//n_head
    self.W_q=nn.Linear(d_model,d_model)
    self.W_k=nn.Linear(d_model,d_model)
    self.W_v=nn.Linear(d_model,d_model)
    self.W_o=nn.Linear(d_model,d_model)
    self.dropout=nn.Dropout(drop_out)

  def split(self,x):
    B,T,_=x.shape
    x=x.view(B,T,self.h,self.dk).transpose(1,2)
    return x

  def forward(self,q,k,v,mask=None):
    q=self.split(self.W_q(q))
    k=self.split(self.W_k(k))
    v=self.split(self.W_v(v))
    out,self.lastatt=scaled_dot_product_attention_1(q,k,v,mask)
    out=self.dropout(out)
    B,H,T,dk=out.shape
    out=out.transpose(1,2).contiguous().view(B,T,H*dk)
    return self.W_o(out)

mha1=multiheadattention(32,4).eval()
x=torch.randn(2,6,32)
causal=torch.tril(torch.ones(6,6,dtype=bool))
y1=mha1(x,x,x,causal)
print(y1.shape)
x2=x.clone();x2[:,4:]=torch.randn(2,2,32)
y2=mha1(x2,x2,x2,causal)
assert torch.allclose(y1[:,:4],y2[:,:4],atol=1e-6)
print("causality OK")

torch.Size([2, 6, 32])
causality OK
